In [ ]:
using Base.Threads
println( "Number of threads: ", nthreads() )

include( "../args.jl" )
include( "../model.jl" )
include( "../geom.jl" )
include( "../recur.jl" )

loadsteps = true
savesteps = !loadsteps
loadsteps, savesteps

In [ ]:
# Case title.
N = 100;  μ = 1.0;  ρ = 0.1;  β = 1.0;  δt = Δt
case = "../data/results/case-2_crit-spd/"
pop = "N-$(N)/mu-$(round( μ, digits=6 ))/rho-$(round( ρ, digits=6 ))_beta-$(round( β, digits=6))/"

if ~isdir( case*pop )
    mkpath( case*pop )
end

In [ ]:
# Length and time scale.
ξ = 0.1;  ϕ = 0.25
scale = Scale( 1.0, ξ )

# Adapatable time-step length.
smin = -3;  smax = 1
slist = round.( 10.0.^(smin:0.25:smax), digits=6 )  # NON-DIMENSIONAL.
Ns = length( slist )
println( "Running for $(Ns) different values of s0." )

# System size and environment density.
L = √(N/μ)

# Activity transition variables.
η = 1/500;  γ = 1/10;  τ = 75.0
λ = 0.50

# Generate parameter variables.
dparams = Params(; ρ=ρ, η=η, β=β, γ=γ, τ=τ, ξ=ξ, λ=λ, ϕ=ϕ, s=2.5 )
paramslist = [adjparams( dparams; s=s/ξ^(ϕ + 1) ) for s ∈ slist]
nondimlist = [Nondim( params; scale ) for params ∈ paramslist];

In [ ]:
# Data folder name.
folderlist = [
    case*pop*"xi-$(round( ξ, digits=6 ))/spd-$(round( s/ξ^(ϕ + 1), digits=6 ))/"
    for s ∈ slist]

In [ ]:
# Run simulation under each environment parameter.
T = round( defInt, 500/scale.T );  M = 50
Tload = round( defInt, 500/scale.T )  # If applicable.

# Compute simulation time-step.
Nt = round( defInt, T/δt );  tlist = 1:Nt
nt = round( defInt, 1/(2*δt*scale.T) );  tsave = Set( 1:nt:Nt )

# Frequency of adjacency calculation.
δt̂ = round( defInt, 0.1/δt );

In [ ]:
# Initialize list and run optimization.
xdatalist = [[Matrix{defFloat}( undef, length( tsave ) + 1, 3 ) for _ ∈ 1:M] for _ ∈ 1:Ns]
zdata = Matrix{State}( undef, Ns, M )
@threads for k ∈ 1:Ns
    nondim = nondimlist[k]
    for m ∈ 1:M
        # If steps are already saved, use as initial state.
        file = loadsteps ? folderlist[k]*"steps/state_T-$(Tload)_m-$(m).txt" : nothing

        # Initialize agent states.
        z = initialstate( N, L; A=1, file=file )
        ẑ = copystate( z )

        # Initialize adjacency and saved state.
        A = proximity( N, L, nondim.r, nondim.α, z.x, z.y, z.θ )
        xdatalist[k][m][1,:] = statecomposition( N, ẑ )

        # Run simulation.
        t̂ = 2
        for t ∈ tlist
            # Update the adjacency matrix.
            (t % δt̂) == 0 && (A = proximity( N, L, nondim.r, nondim.α, z.x, z.y, z.θ ))

            # Step simulation.
            step!( N, L, nondim, z, ẑ; A=A, δt=δt )

            # Save state if in appropriate subset.
            t ∈ tsave && (xdatalist[k][m][t̂,:] = statecomposition( N, ẑ ); t̂ += 1)

            # Swap contents.
            tmp = z;  z = ẑ;  ẑ = tmp
        end

        # Save last simulation state.
        zdata[k,m] = z
    end
end

In [ ]:
# Compute determinism metric and related statistics.
Rdata = Matrix{RecurrenceMap}( undef, Ns, M )
ςdata = Matrix{defFloat}( undef, Ns, M )
@threads for k ∈ 1:Ns
    for m ∈ 1:M
        Rdata[k,m] = recurrence( xdatalist[k][m]; δx=1/100 )
        ςdata[k,m] = determinism( Rdata[k,m]; ℓ0=15 )
    end
end

# Determinism statistics.
ς̄list = vcat( mean( ςdata, dims=2 )... )
ς̄stnd = vcat( std(  ςdata, dims=2 )... );

In [ ]:
# Plot the determinism as a function of the system size.
plt = plot( size=(300,200), xformatter=:plain, dpi=600 )

sc = criticals( dparams, N, μ )*ξ^(ϕ + 1)
plot!( plt, [sc, sc], [0, 1]; color=:gray, lw=2 )
plot!( plt, slist, ς̄list; ribbon=ς̄stnd, color=:black, fillalpha=1/4, lw=2, marker=:circ )

plot!( plt; xlims=(slist[1],slist[end]), xscale=:log10 )
plot!( plt; ylims=(0,1) )

plot!( plt; xlabel="effective temperature, "*L"s_0", ylabel="determinism", legend=false )

In [ ]:
k = Ns-5

# Plot the mean activity deries for each duration value.
plt = plot( size=(400,200), xformatter=:plain, margin=10pt, dpi=600 )

m̂ = 10
for m ∈ 1:M
    alist = xdatalist[k][m][:,1]
    alpha = m == m̂ ? 1 : 1/10
    plot!( plt, δt*(0:nt:Nt), alist; color=:black, alpha=alpha, lw=2, label="" )
end

plot!( plt; xlims=(0,T), xlabel="time, "*L"t" )
plot!( plt; ylims=(0,1), ylabel="proportion of\nants active, "*L"a" )

In [ ]:
for (k, folder) ∈ enumerate( folderlist )
    if !isdir( folder*"steps/" )
        mkpath( folder*"steps/" )
    end

    if true
        xdata = xdatalist[k]
        params = paramslist[k]

        # Save the state variables.
        writedlm( folder*"activity_T-$(round( defInt, T )).txt", [xlist[:,1] for xlist ∈ xdata] )
        writedlm( folder*"inactivity_T-$(round( defInt, T )).txt", [xlist[:,2] for xlist ∈ xdata] )
        writedlm( folder*"refractory_T-$(round( defInt, T )).txt", [xlist[:,3] for xlist ∈ xdata] )
        writedlm( folder*"determinism_T-$(round( defInt, T )).txt", ςdata[k,:] )

        # Save scale and parameters.
        saveparams( folder*"params.json", params )
    end

    if savesteps
        for m ∈ 1:M
            savestate( folder*"steps/state_T-$(T)_m-$(m).txt", zdata[k,m] )
        end
    end
end